# 05c — Convert Mintaka CSV to JSONL with Manual Russian Translations

This notebook prepares a manual translation review table and then builds a final JSONL file from the translations inserted into `question_ru_manual`.

Main rule: after you fill `question_ru_manual`, run only section **6. Build final JSONL from manually translated CSV**. Do not re-run section 5 unless you want to recreate the review table.

## 1. Settings

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import pandas as pd

# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

INPUT_CSV_NAME = "mintaka_reconstruction_shortlist.csv"
REVIEW_CSV_NAME = "mintaka_manual_translation_review_sheet.csv"

OUTPUT_DIR_NAME = "artifacts_mintaka_stage3"
CSV_SUBDIR = "csv"
JSONL_SUBDIR = "jsonl"

DRAFT_JSONL_NAME = "mintaka_selected_manual_translation_draft.jsonl"
FINAL_JSONL_NAME = "mintaka_selected_ru_final.jsonl"

# Column names
ID_COL = "benchmark_id"
QUESTION_EN_COL = "question_en_original"
QUESTION_RU_MANUAL_COL = "question_ru_manual"
QUESTION_RU_COL = "question_ru"
TRANSLATION_STATUS_COL = "translation_status"

# Safety: keep False by default to avoid overwriting your manual CSV.
OVERWRITE_REVIEW_CSV = False


## 2. Helpers

In [10]:
def find_project_file(filename: str) -> Path:
    """Find a file by name under the current directory or its parents."""
    start = Path.cwd().resolve()
    candidates = []

    # Search down from current directory first.
    candidates.extend(start.rglob(filename))

    # Then search under parent directories, which helps when the notebook cwd is nested.
    for parent in [start, *start.parents]:
        maybe = list(parent.glob(f"**/{filename}"))
        candidates.extend(maybe)
        if candidates:
            break

    # Prefer the expected Mintaka stage2 path when present.
    candidates = sorted(set(candidates), key=lambda p: ("artifacts_mintaka_stage2" not in str(p), len(str(p))))
    if not candidates:
        raise FileNotFoundError(f"Could not find {filename!r} from {start}")
    return candidates[0].resolve()


def locate_paths() -> dict:
    input_csv = find_project_file(INPUT_CSV_NAME)
    # Project root is the folder that contains the mintaka directory artifacts, if possible.
    # Expected: <root>/mintaka/artifacts_mintaka_stage2/csv/mintaka_reconstruction_shortlist.csv
    root = input_csv.parents[2] if input_csv.parent.name == "csv" else input_csv.parent

    output_dir = root / OUTPUT_DIR_NAME
    csv_dir = output_dir / CSV_SUBDIR
    jsonl_dir = output_dir / JSONL_SUBDIR
    reports_dir = output_dir / "reports"
    csv_dir.mkdir(parents=True, exist_ok=True)
    jsonl_dir.mkdir(parents=True, exist_ok=True)
    reports_dir.mkdir(parents=True, exist_ok=True)

    return {
        "input_csv": input_csv,
        "output_dir": output_dir,
        "csv_dir": csv_dir,
        "jsonl_dir": jsonl_dir,
        "reports_dir": reports_dir,
        "review_csv": csv_dir / REVIEW_CSV_NAME,
        "draft_jsonl": jsonl_dir / DRAFT_JSONL_NAME,
        "final_jsonl": jsonl_dir / FINAL_JSONL_NAME,
    }


def read_csv_safely(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, dtype=str, keep_default_na=False).fillna("")


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def validate_jsonl(path: Path) -> None:
    if not path.exists():
        print(f"File does not exist: {path}")
        return
    n = 0
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.rstrip("\n")
            if not line:
                raise ValueError(f"Empty line at {line_no}")
            json.loads(line)
            n += 1
    print(f"Valid JSONL: {path}")
    print(f"Lines: {n}")


def infer_question_column(df: pd.DataFrame) -> str:
    preferred = [
        "question_en_original",
        "question",
        "question_text",
        "question_original",
    ]
    for col in preferred:
        if col in df.columns:
            return col
    for col in df.columns:
        if "question" in col.lower() and "translation" not in col.lower() and "ru" not in col.lower():
            return col
    raise ValueError(f"Could not infer English question column. Columns: {list(df.columns)}")


def make_base_rows(source_df: pd.DataFrame) -> list[dict]:
    question_col = infer_question_column(source_df)
    rows = []
    for _, rec in source_df.iterrows():
        item = rec.to_dict()
        benchmark_id = item.get(ID_COL, "")
        question_en = item.get(question_col, "")

        item[ID_COL] = benchmark_id
        item[QUESTION_EN_COL] = question_en
        item.setdefault(QUESTION_RU_MANUAL_COL, "")
        item.setdefault(QUESTION_RU_COL, "")
        item.setdefault(TRANSLATION_STATUS_COL, "needs_manual_translation")

        # Keep the original source row under metadata as well.
        item["metadata"] = {k: v for k, v in rec.to_dict().items()}
        rows.append(item)
    return rows

paths = locate_paths()
for name, path in paths.items():
    print(f"{name}: {path}")


input_csv: /Users/matvey/Desktop/multihop benchmark/mintaka/artifacts_mintaka_stage2/csv/mintaka_reconstruction_shortlist.csv
output_dir: /Users/matvey/Desktop/multihop benchmark/mintaka/artifacts_mintaka_stage3
csv_dir: /Users/matvey/Desktop/multihop benchmark/mintaka/artifacts_mintaka_stage3/csv
jsonl_dir: /Users/matvey/Desktop/multihop benchmark/mintaka/artifacts_mintaka_stage3/jsonl
reports_dir: /Users/matvey/Desktop/multihop benchmark/mintaka/artifacts_mintaka_stage3/reports
review_csv: /Users/matvey/Desktop/multihop benchmark/mintaka/artifacts_mintaka_stage3/csv/mintaka_manual_translation_review_sheet.csv
draft_jsonl: /Users/matvey/Desktop/multihop benchmark/mintaka/artifacts_mintaka_stage3/jsonl/mintaka_selected_manual_translation_draft.jsonl
final_jsonl: /Users/matvey/Desktop/multihop benchmark/mintaka/artifacts_mintaka_stage3/jsonl/mintaka_selected_ru_final.jsonl


## 3. Read source CSV

In [11]:
source_df = read_csv_safely(paths["input_csv"])
print(f"Source rows: {len(source_df)}")
print(f"Columns: {list(source_df.columns)}")
source_df.head(3)


Source rows: 11
Columns: ['benchmark_id', 'source_dataset', 'source_split', 'source_id', 'question_en_original', 'question_translations', 'category', 'complexity_type', 'reasoning_family', 'question_entities', 'answer_type', 'answer_mode', 'answer_text_en', 'answer_entities', 'gold_answers', 'gold_qids', 'supporting_entities', 'supporting_numbers', 'num_answers', 'num_question_entities', 'has_supporting_entities', 'has_supporting_numbers', 'needs_translation', 'needs_rephrase', 'translation_source', 'notes', 'priority_score', 'surface_template', 'is_shallow_bridge', 'is_technical', 'is_binary_question', 'is_count_only', 'temporal_hits', 'comparison_hits', 'intersection_hits', 'extra_constraints', 'reconstructability_score_v2', 'auto_decision', 'decision_reasons']


,benchmark_id,source_dataset,source_split,source_id,question_en_original,question_translations,category,complexity_type,reasoning_family,question_entities,...,is_technical,is_binary_question,is_count_only,temporal_hits,comparison_hits,intersection_hits,extra_constraints,reconstructability_score_v2,auto_decision,decision_reasons
0,mintaka_test_5bd63af8,mintaka,test,5bd63af8,Who were the first two senators to represent t...,{'ar': 'من هما أول عضوان في مجلس الشيوخ اللذان...,politics,multihop,multihop,"[{'name': 'Q15686806', 'entity_type': 'entity'...",...,False,False,False,2,0,0,2,15,keep_for_reconstruction,multi_answer|list_like|temporal_signal
1,mintaka_train_08cbd2e7,mintaka,train,08cbd2e7,Which wives of US presidents were also mothers...,{'ar': 'من هن زوجات رؤساء الجمهورية اللواتي أص...,history,multihop,multihop,"[{'name': 'Q30', 'entity_type': 'entity', 'lab...",...,False,False,False,0,0,2,1,15,keep_for_reconstruction,multi_answer|list_like|multi_constraint_signal
2,mintaka_train_b9354cf6,mintaka,train,b9354cf6,"Which city did the band that wrote ""Shiny Happ...","{'ar': 'في أي مدينة أتت الفرقة التي كتبت ""Shin...",music,multihop,multihop,"[{'name': 'Q2306475', 'entity_type': 'entity',...",...,False,False,False,0,0,2,1,14,keep_for_reconstruction,multi_answer|list_like|multi_constraint_signal


## 4. Build draft JSONL

In [12]:
base_rows = make_base_rows(source_df)
write_jsonl(paths["draft_jsonl"], base_rows)
print(f"Draft JSONL written: {paths['draft_jsonl']}")
validate_jsonl(paths["draft_jsonl"])


Draft JSONL written: /Users/matvey/Desktop/multihop benchmark/mintaka/artifacts_mintaka_stage3/jsonl/mintaka_selected_manual_translation_draft.jsonl
Valid JSONL: /Users/matvey/Desktop/multihop benchmark/mintaka/artifacts_mintaka_stage3/jsonl/mintaka_selected_manual_translation_draft.jsonl
Lines: 11


## 5. Build manual translation review CSV

Run this once to create the CSV template. Then fill `question_ru_manual` manually.

If the review CSV already exists, this cell will not overwrite it unless `OVERWRITE_REVIEW_CSV = True`.

In [13]:
review_csv = paths["review_csv"]

if review_csv.exists() and not OVERWRITE_REVIEW_CSV:
    print(f"Review CSV already exists, keeping it unchanged: {review_csv}")
    review_df = read_csv_safely(review_csv)
else:
    review_rows = []
    for row in base_rows:
        review_rows.append({
            ID_COL: row.get(ID_COL, ""),
            QUESTION_EN_COL: row.get(QUESTION_EN_COL, ""),
            QUESTION_RU_MANUAL_COL: row.get(QUESTION_RU_MANUAL_COL, ""),
            "translation_comment": "",
            "answer_text": row.get("answer_text", row.get("answer_text_original", "")),
            "answer_text_original": row.get("answer_text_original", ""),
            "category": row.get("category", ""),
            "complexity_type": row.get("complexity_type", ""),
            "reasoning_family": row.get("reasoning_family", ""),
            "source_dataset": row.get("source_dataset", "mintaka"),
            "notes": row.get("notes", ""),
        })
    review_df = pd.DataFrame(review_rows)
    review_df.to_csv(review_csv, index=False, encoding="utf-8")
    print(f"Manual translation review CSV written: {review_csv}")

print(f"Review rows: {len(review_df)}")
review_df.head(10)


Manual translation review CSV written: /Users/matvey/Desktop/multihop benchmark/mintaka/artifacts_mintaka_stage3/csv/mintaka_manual_translation_review_sheet.csv
Review rows: 11


,benchmark_id,question_en_original,question_ru_manual,translation_comment,answer_text,answer_text_original,category,complexity_type,reasoning_family,source_dataset,notes
0,mintaka_test_5bd63af8,Who were the first two senators to represent t...,,,,,politics,multihop,multihop,mintaka,
1,mintaka_train_08cbd2e7,Which wives of US presidents were also mothers...,,,,,history,multihop,multihop,mintaka,
2,mintaka_train_b9354cf6,"Which city did the band that wrote ""Shiny Happ...",,,,,music,multihop,multihop,mintaka,
3,mintaka_test_655db184,Who was the pitcher for the winning team in th...,,,,,sports,multihop,multihop,mintaka,
4,mintaka_train_76dc240c,What platforms can the first Animal Crossing m...,,,,,videogames,multihop,multihop,mintaka,
5,mintaka_train_af82fcbd,Who was in charge of the screenplay on the mov...,,,,,books,multihop,multihop,mintaka,
6,mintaka_train_d8a4fd32,What were the two initial continents in the vi...,,,,,videogames,multihop,multihop,mintaka,
7,mintaka_test_7215395f,Where is the 2020 AL MVP from?,,,,,sports,multihop,multihop,mintaka,
8,mintaka_train_2c1804b1,In Kingdom Hearts what is the name of Sora's N...,,,,,videogames,multihop,multihop,mintaka,
9,mintaka_train_fb2caf1b,Where did the winner of the 2017 Nobel Prize i...,,,,,books,multihop,multihop,mintaka,


## 6. Build final JSONL from manually translated CSV

Run this after filling `question_ru_manual` in:

`artifacts_mintaka_stage3/csv/mintaka_manual_translation_review_sheet.csv`

This cell is standalone. It re-reads both CSV files from disk and writes the final JSONL immediately.

In [16]:
from pathlib import Path
import json
import shutil
from datetime import datetime

import pandas as pd


REVIEW_CSV_PATH = Path("artifacts_mintaka_stage3/csv/mintaka_manual_translation_review_sheet.csv")
FINAL_JSONL_PATH = Path("artifacts_mintaka_stage3/jsonl/mintaka_selected_ru_final.jsonl")

ID_COL = "benchmark_id"
QUESTION_EN_COL = "question_en_original"
QUESTION_RU_MANUAL_COL = "question_ru_manual"
QUESTION_RU_COL = "question_ru"

FINAL_JSONL_PATH.parent.mkdir(parents=True, exist_ok=True)

if not REVIEW_CSV_PATH.exists():
    raise FileNotFoundError(f"Review CSV not found: {REVIEW_CSV_PATH}")

review_df = pd.read_csv(REVIEW_CSV_PATH, dtype=str, keep_default_na=False)

required_cols = [ID_COL, QUESTION_EN_COL, QUESTION_RU_MANUAL_COL]
missing_required_cols = [col for col in required_cols if col not in review_df.columns]
if missing_required_cols:
    raise ValueError(
        "Missing required columns in review CSV: "
        + ", ".join(missing_required_cols)
    )

# Normalize translations.
review_df[QUESTION_RU_MANUAL_COL] = (
    review_df[QUESTION_RU_MANUAL_COL]
    .fillna("")
    .astype(str)
    .str.strip()
)

source_rows = len(review_df)

# Rows deleted from the CSV are intentionally ignored.
# Rows with empty translations are skipped.
export_df = review_df[review_df[QUESTION_RU_MANUAL_COL] != ""].copy()

skipped_rows = source_rows - len(export_df)

# Final dataset field.
export_df[QUESTION_RU_COL] = export_df[QUESTION_RU_MANUAL_COL]

final_rows = []

for _, row in export_df.iterrows():
    row_dict = row.to_dict()

    # Clean empty NaN-like values just in case.
    row_dict = {
        key: "" if value is None else value
        for key, value in row_dict.items()
    }

    # Keep both fields:
    # - question_ru_manual: the manually edited translation source
    # - question_ru: the final field used by the benchmark
    row_dict[QUESTION_RU_MANUAL_COL] = row_dict[QUESTION_RU_MANUAL_COL].strip()
    row_dict[QUESTION_RU_COL] = row_dict[QUESTION_RU_COL].strip()
    row_dict["translation_status"] = "manual_translation_ready"

    final_rows.append(row_dict)

if not final_rows:
    raise ValueError(
        "No rows were exported. Check that question_ru_manual is filled "
        f"in {REVIEW_CSV_PATH}."
    )

# Safety backup: do not silently lose the previous final file.
if FINAL_JSONL_PATH.exists():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = FINAL_JSONL_PATH.with_suffix(f".backup_{timestamp}.jsonl")
    shutil.copy2(FINAL_JSONL_PATH, backup_path)
    print(f"Existing final JSONL backed up to: {backup_path}")

with FINAL_JSONL_PATH.open("w", encoding="utf-8") as f:
    for row in final_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Review CSV used: {REVIEW_CSV_PATH}")
print(f"Rows in review CSV: {source_rows}")
print(f"Exported rows: {len(final_rows)}")
print(f"Skipped rows with empty translations: {skipped_rows}")
print(f"Final JSONL written: {FINAL_JSONL_PATH}")

pd.DataFrame(final_rows)[[ID_COL, QUESTION_EN_COL, QUESTION_RU_COL]].head(10)

Review CSV used: artifacts_mintaka_stage3/csv/mintaka_manual_translation_review_sheet.csv
Rows in review CSV: 9
Exported rows: 9
Skipped rows with empty translations: 0
Final JSONL written: artifacts_mintaka_stage3/jsonl/mintaka_selected_ru_final.jsonl


,benchmark_id,question_en_original,question_ru
0,mintaka_test_5bd63af8,Who were the first two senators to represent t...,"Кто были первыми двумя сенаторами, представляв..."
1,mintaka_train_08cbd2e7,Which wives of US presidents were also mothers...,Какие жёны президентов США также были матерями...
2,mintaka_train_b9354cf6,"Which city did the band that wrote ""Shiny Happ...","Из какого города была группа, написавшая песню..."
3,mintaka_test_655db184,Who was the pitcher for the winning team in th...,Кто был питчером победившей команды в первой М...
4,mintaka_train_af82fcbd,Who was in charge of the screenplay on the mov...,"Кто отвечал за сценарий фильма, снятого по тре..."
5,mintaka_train_d8a4fd32,What were the two initial continents in the vi...,Какие два континента изначально были доступны ...
6,mintaka_train_2c1804b1,In Kingdom Hearts what is the name of Sora's N...,Как зовут лучших друзей Nobody Соры в Kingdom ...
7,mintaka_train_fb2caf1b,Where did the winner of the 2017 Nobel Prize i...,В каком университете учился лауреат Нобелевско...
8,mintaka_train_5362eb29,What is the smallest country that is part of t...,Какая самая маленькая страна находится на само...


## 7. Quick JSONL validation

In [17]:
paths = locate_paths()
validate_jsonl(paths["draft_jsonl"])
validate_jsonl(paths["final_jsonl"])


Valid JSONL: /Users/matvey/Desktop/multihop benchmark/mintaka/artifacts_mintaka_stage3/jsonl/mintaka_selected_manual_translation_draft.jsonl
Lines: 9
Valid JSONL: /Users/matvey/Desktop/multihop benchmark/mintaka/artifacts_mintaka_stage3/jsonl/mintaka_selected_ru_final.jsonl
Lines: 9
